# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [14]:
# The rule: A content piece is worth reviewing if it is moderately-to-very stale (31 days or more since its last update) and its CTR is weaker than expected for its current search position. Staleness alone is a mixed signal (it rises with age but reverses at the oldest tier, likely due to a small sample there), so I only flag content where staleness AND weak CTR-for-position occur together, which is a stronger combined signal than either alone.
# Reason codes it can output: stale_and_weak_ctr (both conditions true), stale_only, weak_ctr_only, no_flag.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
import numpy as np

# Signal flags (transparent, no fitted weights)
stale = (df["days_since_last_update"] >= 31).astype(int)

# expected CTR for this position tier, then flag CTR below that expectation
expected_ctr_by_tier = df.groupby("position_tier")["ctr"].transform("mean")
weak_ctr = (df["ctr"] < expected_ctr_by_tier).astype(int)

# Score: stale AND weak_ctr, scaled by recent visibility (impressions) so that
# high-traffic weak content ranks above low-traffic weak content
df["score"] = stale * weak_ctr * df["impressions_last_30d"]

# Reason codes
conditions = [
    (stale == 1) & (weak_ctr == 1),
    (stale == 1) & (weak_ctr == 0),
    (stale == 0) & (weak_ctr == 1),
]
choices = ["stale_and_weak_ctr", "stale_only", "weak_ctr_only"]
df["reason_code"] = np.select(conditions, choices, default="no_flag")

# Action label
df["action"] = np.where(df["score"] > 0, "refresh_title_and_opening", "no_action")

# Rank and write the queue
df_sorted = df.sort_values("score", ascending=False)
df_sorted.to_csv("work/outputs/baseline_action_score.csv", index=False)

# Evaluate with precision@K (using is_declining as the label, for evaluation only,
# never as a score input)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [10, 20, 50]:
    p_at_k = precision_at_k(df_sorted["score"], df_sorted["is_declining"], k)
    base_rate = df["is_declining"].mean()
    print(f"precision@{k}: {p_at_k:.3f}  |  base rate: {base_rate:.3f}")

print(df_sorted["reason_code"].value_counts())

OSError: Cannot save file into a non-existent directory: 'work/outputs'

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
#content_2cb567c3c89b (pos 22.2, ctr 0.10, 48 days stale) — Action: refresh title/opening. Why: stale + weak CTR combo. What would make it wrong: position 22 is naturally low-CTR territory regardless of content quality, so this might just be reflecting position, not a real fixable problem.
#content_2dba2b1f9536 (pos 27.9, ctr 0.21, 104 days) — Same pattern as #1. What would make it wrong: same caveat — deep position may explain low CTR on its own.
#content_4a6607efcb46 (pos 2.2, ctr 0.01, 104 days) — Strongest pick in the list: top-3 position but essentially zero clicks, a real mismatch worth investigating. What would make it wrong: this could be a tracking/data artifact (broken snippet, very new clicks not yet accrued) rather than an actual title problem.
#content_5fe46e04994d (pos 4.2, ctr 0.14, 104 days) — Similar to #3: strong position, weak CTR, genuinely worth a look. What would make it wrong: if the query is low-intent, low CTR at this position could be normal for the topic.
#content_9532f197bbc8 (pos 2.0, ctr 0.87, 104 days) — Weakest justification in the top 10: CTR is close to the top_3 tier average (~1.48), not dramatically low. What would make it wrong: this might not deserve top-priority refresh at all, since 0.87 isn't clearly broken.
#content_36ff89c8214e (pos 7.3, ctr 0.05, 104 days) — Reasonable flag: decent position, very low CTR. What would make it wrong: if main_intent is informational, low CTR may just be typical for that intent type.
#content_2c2606c5d176 (pos 4.2, ctr 0.53, 104 days) — Borderline like #5: CTR isn't dramatically below expectation for this tier. What would make it wrong: may be flagged prematurely if 0.53 is within normal range for its tier.
#content_b28d1efd668f (pos 26.2, ctr 0.06, 104 days) — Same deep-position pattern as #1/#2. What would make it wrong: low CTR likely explained by position alone.
#content_f02b48f88241 (pos 25.8, ctr 0.10, 104 days) — Same as above.
#content_91652435f57a (pos 7.8, ctr 0.06, 104 days) — Decent position, low CTR, real mismatch worth flagging. What would make it wrong: a SERP feature (image pack, snippet elsewhere) could be suppressing clicks regardless of title quality.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
##5 (content_9532f197bbc8, ctr 0.87 at position 2.0) and #7 (content_2c2606c5d176, ctr 0.53 at position 4.2) are the shakiest picks in the top 10. Both sit reasonably close to their tier's average CTR (top_3 tier average is ~1.48, so 0.87 isn't dramatically below it), unlike the clear mismatches in #3 and #4 (ctr near 0.01–0.14 at top positions). Flagging these two as "weak CTR" is a stretch — a future version of this rule should require CTR to fall further below the tier average, not just any amount below it, to avoid diluting the queue with borderline cases.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.